In [20]:
import duckdb
import pandas as pd
from pathlib import Path

con = duckdb.connect("../instacart.duckdb")

In [21]:
tables = con.execute("SHOW TABLES").df()
tables = tables["name"]

for table in tables:
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(table, count)

aisles 134
departments 21
order_products_prior 32434489
order_products_train 1384617
orders 3421083
products 49688


In [28]:
con.execute("""
            select table_name, count(*) as column_count
            from information_schema.columns
            group by table_name
            """).df()

,table_name,column_count
0,order_products_train,4
1,aisles,2
2,departments,2
3,products,4
4,orders,7
5,order_products_prior,4


In [79]:

def check_null(table_name):
    columns = con.execute(f"""
            select column_name
            from information_schema.columns
            where table_name = '{table_name}'
            """).df()
    columns = columns['column_name']
    
    query = """
        SELECT
        COUNT(*) AS total_rows, \n
        """
    
    for column in columns:
        query += f"COUNT(*) - COUNT({column}) AS {column}_NULL, \n"

    query = query[:-2] + f"\nFROM {table_name}"

    df = con.execute(query).df()
    df["table_name"] = table_name

    return df

for table in tables:
    print(check_null(table))

   total_rows  aisle_id_NULL  aisle_NULL table_name
0         134              0           0     aisles
   total_rows  department_id_NULL  department_NULL   table_name
0          21                   0                0  departments
   total_rows  order_id_NULL  product_id_NULL  add_to_cart_order_NULL  \
0    32434489              0                0                       0   

   reordered_NULL            table_name  
0               0  order_products_prior  
   total_rows  order_id_NULL  product_id_NULL  add_to_cart_order_NULL  \
0     1384617              0                0                       0   

   reordered_NULL            table_name  
0               0  order_products_train  
   total_rows  order_id_NULL  user_id_NULL  eval_set_NULL  order_number_NULL  \
0     3421083              0             0              0                  0   

   order_dow_NULL  order_hour_of_day_NULL  days_since_prior_order_NULL  \
0               0                       0                       206209 

In [81]:
con.close()